# 05 - Silver Validation
### Goal
- validate all silver tables
- check nulls, negative values, schema consistency, key field quality

In [0]:
%run ../01_setup/00_config

In [0]:
import logging
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

## Expected schemas and key columns

In [0]:
expected_schemas = {
    silver_market_prices_table: {
        "required_columns": {
            "event_date", "region", "market_type",
            "price_eur_mwh", "volume_mwh", "source_system",
            "last_updated_ts", "is_high_price", "price_band"
        },
        "key_columns":      ["event_date", "region", "price_eur_mwh"],
        "numeric_checks":   [("price_eur_mwh", 0), ("volume_mwh", 0)]
    },
    silver_weather_table: {
        "required_columns": {
            "event_date", "region", "temperature_c",
            "wind_speed_kmh", "precipitation_mm",
            "weather_alert_level", "source_system",
            "last_updated_ts", "is_severe_weather", "wind_class"
        },
        "key_columns":      ["event_date", "region"],
        "numeric_checks":   [("wind_speed_kmh", 0), ("precipitation_mm", 0)]
    },
    silver_grid_events_table: {
        "required_columns": {
            "event_id", "event_date", "region", "asset_id",
            "event_type", "severity", "duration_minutes",
            "source_system", "last_updated_ts",
            "is_critical_event", "duration_band"
        },
        "key_columns":      ["event_id", "event_date", "region"],
        "numeric_checks":   [("duration_minutes", 0)]
    },
    silver_integrated_table: {
        "required_columns": {
            "event_date", "region", 
            "avg_price_eur_mwh", "avg_temperature_c", "avg_wind_speed_kmh",
            "total_volume_mwh"
        },
        "key_columns":      ["event_date", "region"],
        "numeric_checks":   [("avg_price_eur_mwh", 0)],
        "numeric_checks":   [("avg_temperature_c", 0)],
        "numeric_checks":   [("avg_wind_speed_kmh", 0)]
    }
}

## Run validation checks

In [0]:
validation_passed = True
results = []

for table, config in expected_schemas.items():
    log.info(f"--- Validating: {table} ---")

    # table existence
    try:
        df = spark.table(table)
        log.info(f"[EXISTS]  {table} found.")
    except AnalysisException:
        log.error(f"[MISSING] Table not found: {table}")
        validation_passed = False
        results.append({"table": table, "check": "existence", "status": "FAILED", "value": ""})
        continue

    # row count
    row_count = df.count()
    if row_count > 0:
        log.info(f"[ROWS]    {table} has {row_count} rows.")
        results.append({"table": table, "check": "row_count", "status": "PASSED", "value": str(row_count)})
    else:
        log.error(f"[ROWS]    {table} is empty.")
        validation_passed = False
        results.append({"table": table, "check": "row_count", "status": "FAILED", "value": "0"})

    # schema consistency
    actual_columns  = set(df.columns)
    missing_columns = config["required_columns"] - actual_columns
    if missing_columns:
        log.error(f"[SCHEMA]  {table} missing columns: {missing_columns}")
        validation_passed = False
        results.append({"table": table, "check": "schema", "status": "FAILED", "value": str(missing_columns)})
    else:
        log.info(f"[SCHEMA]  {table} schema OK.")
        results.append({"table": table, "check": "schema", "status": "PASSED", "value": ""})

    # key field nulls
    for key_col in config["key_columns"]:
        null_count = df.filter(F.col(key_col).isNull()).count()
        if null_count > 0:
            log.error(f"[NULLS]   {table}.{key_col} has {null_count} null values.")
            validation_passed = False
            results.append({"table": table, "check": f"nulls_{key_col}", "status": "FAILED", "value": str(null_count)})
        else:
            log.info(f"[NULLS]   {table}.{key_col} OK.")
            results.append({"table": table, "check": f"nulls_{key_col}", "status": "PASSED", "value": "0"})

    # negative value checks
    for col_name, min_val in config["numeric_checks"]:
        neg_count = df.filter(F.col(col_name) < min_val).count()
        if neg_count > 0:
            log.error(f"[NEGATIVE] {table}.{col_name} has {neg_count} values below {min_val}.")
            validation_passed = False
            results.append({"table": table, "check": f"negative_{col_name}", "status": "FAILED", "value": str(neg_count)})
        else:
            log.info(f"[NEGATIVE] {table}.{col_name} OK.")
            results.append({"table": table, "check": f"negative_{col_name}", "status": "PASSED", "value": "0"})

## Validation summary

In [0]:
summary_df = spark.createDataFrame(results)
display(summary_df)

if not validation_passed:
    raise ValueError("Silver validation failed. Check the summary above for details.")
else:
    log.info("All silver validation checks passed.")